# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will walk through record set exploration, field inspection, and data analysis using strong referencing via Croissant `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets, their `@id`s, and information on fields for later referencing. This is crucial for addressing individual entities with their `@id`, which ensures robust, schema-consistent data handling.

In [ ]:
# The dataset may contain one or more record sets. See their @id and fields.
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSets', [])] if hasattr(metadata, 'recordSets') else []
if not record_set_ids:
    # Fallback for Croissant versions where it's singular or in different attr
    record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])] if hasattr(metadata, 'recordSet') else []

print("Available Record Sets (by @id):")
for rsid in record_set_ids:
    print(f"- {rsid}")

# Show available fields per record set
for rsid in record_set_ids:
    print(f"\nFields for record set '@id': {rsid}")
    # Access the full record set definition
    record_set_obj = next((rs for rs in getattr(metadata, 'recordSets', getattr(metadata, 'recordSet', [])) if rs['@id'] == rsid), None)
    if record_set_obj and 'field' in record_set_obj:
        for field in record_set_obj['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', field.get('@id'))}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

**Note on Referencing:**
When extracting data, always reference record sets and fields by their `@id`. Determine the target record set and field from the previous overview step.

In [ ]:
# List record set @ids for reference
# If there is only one, use it for extraction
record_sets = record_set_ids
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Extracted DataFrame for record set: {record_set} -- shape: {dataframes[record_set].shape}")
    print(f"Columns (@id): {dataframes[record_set].columns.tolist()}")
    display(dataframes[record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All references to columns/fields use their `@id` as previously displayed.

Pick a numeric field (e.g., normalized intensity, molecular weight, or peptide count, referenced by its `@id` as found above).

In [ ]:
# Example: Let's pick fields by their actual @id here.
import numpy as np

# If there are multiple record sets, use the main data table (replace with actual @id as per section 2 output)
main_record_set_id = record_sets[0] if record_sets else None
main_df = dataframes[main_record_set_id]

# Example: Replace with real @id as found from section 2
# Suppose main numeric field's @id is 'cr:protein_mw' and group field is 'cr:sample_type'
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    if 'mw' in col.lower() or 'molecular_weight' in col.lower():
        numeric_field_id = col
    if 'sample' in col.lower() and 'type' in col.lower():
        group_field_id = col

if numeric_field_id is None:
    # Fallback: choose first numeric-like column
    numeric_field_id = main_df.select_dtypes(include=[np.number]).columns[0] if not main_df.select_dtypes(include=[np.number]).empty else main_df.columns[0]
if group_field_id is None:
    # Fallback: choose a likely categorical field
    group_field_id = main_df.columns[1] if len(main_df.columns) > 1 else main_df.columns[0]

print(f"Using numeric field @id for analysis: {numeric_field_id}")
print(f"Using group field @id for analysis: {group_field_id}")

# Remove missing values for the numeric field and filter
threshold = main_df[numeric_field_id].mean() if np.issubdtype(main_df[numeric_field_id].dtype, np.number) else 10
filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"Filtered records (N={len(filtered_df)}) with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
if group_field_id in main_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset referencing them by their `@id`.

Below is an example histogram of the normalized numeric field, and a boxplot by group, if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel('Count')
plt.show()

# Boxplot grouped by group_field, if available
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=f"{numeric_field_id}_normalized", data=filtered_df)
    plt.title(f"Normalized {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² Croissant dataset on Mass Spectrometry Analysis of Extracellular Vesicles from human mast cells. 

- All data entities (record sets, fields) were referenced by their Croissant `@id` throughout.
- Typical EDA operations (filtering, normalization, grouping) and visualizations were performed.
  
Proceed to more advanced analysis or integration with other tools as needed. For more, see: [mlcroissant docs](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant#readme)